In [2]:
print("ok")

ok


In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Naveen Yadav Golla\\MLops\\chatbot\\Building_chatbot_with_LLM'

In [1]:
# ===============================
# WealthGPT: Financial RAG Chatbot
# ===============================

# 1️⃣ Imports
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 2️⃣ Load environment variables
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PINECONE_API_KEY or not OPENAI_API_KEY:
    raise ValueError("❌ Missing API keys in environment variables.")

# 3️⃣ Pinecone setup
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "financial-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  # matches all-MiniLM-L6-v2 output
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# 4️⃣ Load embedding model
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 5️⃣ PDF Loader and Chunking
def load_and_chunk_pdfs(folder_path: str, chunk_size=1000, chunk_overlap=200):
    folder_path = Path(folder_path)
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    loader = DirectoryLoader(str(folder_path), glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} PDF documents")

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(documents)
    print(f"✅ Split into {len(chunks)} chunks")
    return chunks

texts_chunk = load_and_chunk_pdfs("data")

# 6️⃣ Initialize Pinecone VectorStore
def init_vectorstore(documents=None):
    if documents:
        return PineconeVectorStore.from_documents(
            documents=documents,
            embedding=embedding,
            index_name=index_name
        )
    return PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )

docstore = init_vectorstore(documents=texts_chunk)

# 7️⃣ Retriever
retriever = docstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

# 8️⃣ Chat Model
chat_model = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)

# 9️⃣ History-aware retriever (reformulates follow-up questions as standalone)
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, "
     "reformulate it as a standalone question. Do NOT answer it."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(chat_model, retriever, contextualize_q_prompt)

# 🔟 Financial RAG Prompt with conversation history
system_prompt = (
    "You are WealthGPT, an AI assistant for financial decision-making. "
    "Use the retrieved context below to answer the user's question. "
    "If the exact answer is not present, summarize related information from the context. "
    "Provide clear, accurate, and concise financial guidance. "
    "Do not hallucinate beyond the provided documents. "
    "Keep your response under three sentences.\n\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# 1️⃣1️⃣ RAG chain
qa_chain = create_stuff_documents_chain(chat_model, prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

# 1️⃣2️⃣ Session-based message history store
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# 1️⃣3️⃣ Interactive Chat Loop
print("💬 WealthGPT is ready! Type 'exit' to quit.\n")
session_id = "user_session_1"

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit", "bye"):
        print("WealthGPT: Goodbye! Stay financially savvy. 👋")
        break
    response = conversational_rag.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"WealthGPT: {response['answer']}\n")

c:\ProgramData\anaconda3\envs\Building_chatbot_with_LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Naveen Yadav Golla\AppData\Local\Temp\ipykernel_676\248554639.py:43: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


FileNotFoundError: Folder not found: data